# 04 - Kontrol loop penuh

Robot bergerak. Pastikan area aman. Tombol **E-STOP** selalu terlihat.

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import ipywidgets as W
from IPython.display import display

from robot.config import Config

In [ ]:
cfg = Config()
cfg.port = "COM5"          # None / "" = dry-run (tanpa robot)
cfg.model = "yolo11n.pt"   # pretrained sudah cukup untuk uji coba
cfg.mode = "stop"          # ada objek -> robot berhenti
cfg.target_class = None    # None = objek apa saja; "person" = manusia saja
cfg.imgsz = 416
cfg

In [ ]:
from robot.brain import Brain

brain = Brain(cfg)
brain.start()

In [ ]:
preview = W.Image(format="jpeg")
status  = W.HTML()
estop   = W.Button(description="E-STOP", button_style="danger",
                   layout=W.Layout(width="120px", height="60px"))
display(W.HBox([preview, W.VBox([estop, status])]))

estop.on_click(lambda b: brain.stop())

async def tick():
    while brain.running:
        jpg = brain.latest_jpeg()
        if jpg:
            preview.value = jpg
        s = brain.state
        status.value = (f"cmd={s['cmd']} (raw={s['raw']})<br>"
                        f"fps={s['fps']:.1f}<br>"
                        f"det={s['det']}<br>err={s['err']}")
        await asyncio.sleep(0.1)

ui = asyncio.create_task(tick())

In [ ]:
ui.cancel()
brain.stop()